# 🎫 Project Brief — NimbusTech Support Ticket Triage & Escalation API

**This notebook is a reference document, not a workbook.** Read it top to bottom to understand the client's requirements, then build the actual solution in **VS Code** as a proper multi-file Python project. Do not write your solution inside this notebook.

**What you'll apply:**
- Scripting Best Practices (structure, SRP, constants, DRY, error handling, logging, docstrings)
- PEP8 (naming, formatting, imports)
- A **basic** FastAPI service (simple routes only — no database, no auth, no advanced FastAPI features; those come later)


## 📖 Client Scenario

**Client:** NimbusTech Solutions — a B2B SaaS company providing project-management software to ~200 corporate clients.

**Background:** NimbusTech's support team uses an old legacy helpdesk tool that can only export tickets as a raw CSV file, once every night, to a shared folder. Every morning, a support-ops analyst manually opens this CSV in Excel to figure out:

- Which tickets have **breached their SLA** (Service Level Agreement — the promised response time)
- Which tickets are **high priority** and need immediate attention
- Which rows in the export are **broken or incomplete** (the legacy tool occasionally exports corrupted rows)

This manual review takes the analyst roughly 45 minutes every single morning, and human error means SLA breaches are sometimes missed until a client complains.

**The ask:** NimbusTech's engineering manager has asked your team (the new intern batch) to build an **MVP (Minimum Viable Product)** with two connected pieces:

1. A **command-line processing script** that reads the nightly CSV export, validates it, classifies every ticket, and writes a clean structured report.
2. A **basic internal FastAPI service** that the support-ops dashboard (built by a different team) can call over HTTP to fetch ticket data and make simple updates — replacing "someone emails a CSV around" with "the dashboard just calls an API."

> ⚠️ **Explicitly out of scope for this MVP:** a real database, authentication/login, or any advanced FastAPI features (dependency injection, middleware, background tasks, etc.). Keep the FastAPI part deliberately simple — those concepts are covered in later sessions. The client only wants a working, basic, reliable MVP right now.


## ✅ Functional Requirements — Module 1: Ticket Processing Script

| ID | Requirement |
|----|-------------|
| FR1 | Read the raw tickets CSV with columns: `ticket_id, customer_name, category, priority_raw, created_at, sla_hours, status` |
| FR2 | Validate every row — a row is **invalid** if `ticket_id` is missing, `sla_hours` is not a positive number, `created_at` is not a parseable date, or `priority_raw` is not one of `low / medium / high / critical` |
| FR3 | Invalid rows must be **skipped and collected separately** (never silently dropped, never crash the script) |
| FR4 | For every valid ticket, compute whether it has **breached its SLA**: breached if `created_at + sla_hours` (in hours) is earlier than "now" AND `status` is not `closed` |
| FR5 | Assign a **priority score** to every valid ticket using a simple mapping (e.g. `low=1, medium=2, high=3, critical=4`) — used later for sorting |
| FR6 | Compute summary counts: total tickets processed, total valid, total invalid, total breached, breakdown by `category` |
| FR7 | If the **invalid-row ratio exceeds 10%** of all rows, the script must **abort processing entirely** (do not write a report) and exit with a non-zero exit code — this signals that the upstream export itself is broken and shouldn't be trusted |
| FR8 | Write a clean structured **JSON report** (see shape below) to an output path |
| FR9 | The script must be runnable from the command line with the input CSV path passed as an argument (use `argparse`), e.g. `python main.py --input data/tickets_raw.csv --output data/tickets_report.json` |
| FR10 | Every meaningful step (start, rows read, rows skipped, abort triggered, report written) must be **logged**, not printed |


## ✅ Functional Requirements — Module 2: Basic FastAPI Service

Keep this **simple** — plain routes, simple Python types / a small Pydantic model, JSON responses. No database required: it's fine to load the processed report into memory (a list/dict) when the app starts, and keep updates in memory for this MVP.

| ID | Requirement |
|----|-------------|
| FR11 | `GET /` — basic root endpoint, returns a simple status/welcome message |
| FR12 | `GET /tickets` — return the full list of processed tickets |
| FR13 | `GET /tickets/breached` — return only tickets that breached their SLA |
| FR14 | `GET /tickets/{ticket_id}` — return one ticket by id; return a `404` if it doesn't exist |
| FR15 | `POST /tickets` — add a new ticket record (accept the core fields via a simple request body) |
| FR16 | `PUT /tickets/{ticket_id}` — update an existing ticket's `status` and/or `priority_raw` |
| FR17 | `DELETE /tickets/{ticket_id}` — remove a ticket record |
| FR18 | On startup, the API should load data from the JSON report produced by Module 1 (read the file at startup — no need for anything fancier) |


## 🛠 Non-Functional Requirements

| ID | Requirement |
|----|-------------|
| NFR1 | Code must follow **PEP8** — naming conventions, spacing, import grouping, line length. Feel free to self-check with `black` / `ruff` |
| NFR2 | Script must follow professional structure — constants at the top, small single-purpose functions (SRP), a `main()` function, and the `if __name__ == "__main__":` guard |
| NFR3 | **No hardcoding** — SLA breach threshold, invalid-row abort ratio, priority mapping, and file paths must all be constants or CLI arguments, not magic values buried in logic |
| NFR4 | Apply **DRY** — no repeated logic; extract shared behavior into functions |
| NFR5 | Use `logging`, not `print()`, for all operational output |
| NFR6 | Handle errors gracefully — a missing file, an empty CSV, or a fully corrupted export must produce a clear log message and a clean exit, never a raw traceback |
| NFR7 | Every function must have a **docstring** describing what it does, its parameters, and its return value |
| NFR8 | Use a **virtual environment** with a `requirements.txt` listing exact dependencies |
| NFR9 | The FastAPI service must stay **basic** — no database, no authentication, no middleware, no dependency injection chains. Simplicity is a requirement here, not a shortcut |
| NFR10 | Organize the project into **multiple files** by responsibility (see suggested structure below) — no single giant script or single giant `main.py` for the whole thing |


## 📄 Sample Input Data

You can use this as your `tickets_raw.csv` sample (create the file yourself in VS Code — a few rows are intentionally invalid, to test your validation logic):


```csv
ticket_id,customer_name,category,priority_raw,created_at,sla_hours,status
T-1001,Acme Corp,billing,high,2026-07-20 09:00:00,24,open
T-1002,Globex Inc,technical,critical,2026-07-19 14:30:00,8,open
T-1003,Initech,technical,medium,2026-07-21 10:00:00,48,closed
T-1004,,billing,low,2026-07-20 11:00:00,72,open
T-1005,Umbrella LLC,technical,critical,2026-07-18 08:00:00,-5,open
T-1006,Hooli,onboarding,medium,not-a-date,24,open
T-1007,Stark Industries,billing,unknown,2026-07-21 09:00:00,24,open
T-1008,Wayne Enterprises,technical,high,2026-07-19 09:00:00,24,open
```


Row `T-1004` (missing customer name — treat as invalid), `T-1005` (negative SLA hours), `T-1006` (bad date), and `T-1007` (invalid priority value) should all be caught by your validation and routed to the invalid-rows list, not crash your script.


## 📦 Expected JSON Report Shape (output of Module 1, input to Module 2)

Your `tickets_report.json` should look roughly like this — exact field names are up to you, but the *shape/information* should match:


```json
{
  "generated_at": "2026-07-23T09:15:00",
  "summary": {
    "total_rows": 8,
    "valid_tickets": 4,
    "invalid_rows": 4,
    "breached_count": 2,
    "by_category": { "billing": 1, "technical": 3 }
  },
  "tickets": [
    {
      "ticket_id": "T-1001",
      "customer_name": "Acme Corp",
      "category": "billing",
      "priority_raw": "high",
      "priority_score": 3,
      "created_at": "2026-07-20T09:00:00",
      "sla_hours": 24,
      "status": "open",
      "sla_breached": true
    }
  ],
  "invalid_rows": [
    { "raw_row": "T-1004,,billing,low,2026-07-20 11:00:00,72,open", "reason": "missing customer_name" }
  ]
}
```


## 🚀 FastAPI Starter Skeleton (reference only — flesh this out yourself)

This is the *only* code you're given — just to confirm the basic style expected. Everything else (loading the report, the remaining routes, the Pydantic model for `POST`/`PUT`) is for you to build.


In [1]:
from fastapi import FastAPI, HTTPException

app = FastAPI(title="NimbusTech Ticket Triage API")


@app.get("/")
def read_root():
    """Basic health/status endpoint."""
    return {"status": "ok", "service": "ticket-triage-api"}


# TODO: GET /tickets
# TODO: GET /tickets/breached
# TODO: GET /tickets/{ticket_id}  (raise HTTPException(404, ...) if missing)
# TODO: POST /tickets
# TODO: PUT /tickets/{ticket_id}
# TODO: DELETE /tickets/{ticket_id}


## 🗂 Suggested Project Structure

Build this in VS Code as a real multi-file project — not a single script:

```
nimbustech_ticket_api/
├── venv/                        (not committed)
├── requirements.txt
├── README.md
├── data/
│   ├── tickets_raw.csv
│   └── tickets_report.json      (generated by the script)
├── ticket_processor/
│   ├── __init__.py
│   ├── config.py                (constants: SLA thresholds, abort ratio, priority map)
│   ├── models.py                (Ticket data structure, e.g. a dataclass)
│   ├── validators.py            (row validation logic)
│   ├── processor.py             (classification + report building)
│   └── main.py                  (argparse + main() + logging setup)
└── ticket_api/
    ├── __init__.py
    ├── main.py                  (FastAPI app + routes)
    ├── models.py                (Pydantic request/response models)
    └── store.py                 (loads report JSON, holds in-memory ticket list)
```


## 🧭 Suggested Build Order (milestones)

1. **Milestone 1 — Processing script.** Build `ticket_processor/` end to end. Run it against the sample CSV from the command line and confirm `tickets_report.json` is generated correctly. Test that the abort-on-bad-data rule (FR7) actually triggers if you feed it a CSV where more than 10% of rows are invalid.
2. **Milestone 2 — Basic API.** Build `ticket_api/`, load the report at startup, implement all six routes. Test every route manually via `http://127.0.0.1:8000/docs`.
3. **Milestone 3 — Polish.** Clean up naming/formatting for PEP8, make sure every function has a docstring, double-check no hardcoded values remain, and write a short `README.md` explaining how to run both parts.


## 🎯 Acceptance Criteria (Definition of Done)

- [ ] Running `python -m ticket_processor.main --input data/tickets_raw.csv --output data/tickets_report.json` produces a correct, well-formed report
- [ ] Feeding a CSV with >10% invalid rows causes the script to abort cleanly with a non-zero exit code and a clear log message (no report written)
- [ ] `uvicorn ticket_api.main:app --reload` starts the service and all 6 routes work correctly when tested via `/docs`
- [ ] `GET /tickets/{id}` returns `404` for an id that doesn't exist
- [ ] No hardcoded file paths, thresholds, or magic numbers anywhere in either module
- [ ] Every function has a docstring
- [ ] Code is organized across multiple files as shown above — not one giant file
- [ ] Code is PEP8-clean (check with `black --check .` and `ruff check .`)


## 📝 Evaluation Rubric (how this will be graded)

| Area | What's checked |
|------|-----------------|
| Correctness | Does the script classify/validate tickets correctly? Do all 6 API routes behave correctly? |
| Scripting best practices | SRP, constants, DRY, `main()` guard, docstrings, no hardcoding |
| PEP8 / readability | Naming, formatting, import grouping, meaningful comments |
| Error handling & logging | Graceful failure on bad input; proper use of `logging` over `print()` |
| Project structure | Multi-file organization matching responsibilities, not a monolith |
| FastAPI simplicity | Solution stays basic as required — no unnecessary complexity added |


## 🌟 Stretch Goals (optional — only if you finish early)

- Add a `GET /tickets/category/{category}` filter endpoint
- Add a `GET /tickets/summary` endpoint that returns the `summary` block from the report
- Sort `GET /tickets` results by `priority_score` descending
- Add a simple `PATCH /tickets/{ticket_id}` for partial updates (compare this to `PUT` — what's the real difference in your implementation?)

Keep these optional and don't let them delay finishing the core requirements above first.
